# 05. Margens de segurança e contexto para a decisão

O artefato apresenta três opções para cada dia e os dados que ajudam a escolher. **A decisão é da nutricionista e da gestora.**

- **Sem margem:** usa a previsão como está.
- **Margem moderada:** calculada para haver déficit potencial em cerca de 1 a cada 4 dias.
- **Margem alta:** calculada para que isso ocorra em cerca de 1 a cada 10 dias.

As margens de segurança vêm dos erros dos últimos 60 dias já conhecidos em cada sexta-feira do planejamento. Excesso e déficit são potenciais: o restaurante não pesa o que é descartado nem registra quem ficou sem refeição.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

sys.path.append("../src")
from nomes import FOLGAS, UNIVERSOS

pd.set_option("display.max_rows", 60, "display.max_columns", 20, "display.width", 180)
SAIDA = Path("..") / "resultados" / "planejamento"
v = pd.read_csv(SAIDA / "verificacao_folgas_2026.csv", sep=";", encoding="utf-8-sig")
s = pd.read_csv(SAIDA / "verificacao_por_situacao_2026.csv", sep=";", encoding="utf-8-sig")
NOMES_OPCAO = {**FOLGAS, "nutricionista": "Quantitativo da nutricionista"}

## 1. As margens de segurança protegem como prometem?

A coluna **prometido** é a estimativa feita antes de cada semana; **observado** é o que aconteceu em 2026, sem dias atípicos.

In [2]:
d = v[v["universo"] == "sem_atipicos"].copy()
d["Opção"] = d["opcao"].map(NOMES_OPCAO)
d = d.rename(columns={"decisao": "Decisão", "folga_media": "Margem média", "prometido_dias_a_menos_pct": "Déficit potencial, prometido (% dos dias)",
                      "dias_a_menos_pct": "Déficit potencial, observado (% dos dias)", "a_mais": "Excesso potencial (refeições)", "a_menos": "Déficit potencial (refeições)"})
d[["Decisão", "Opção", "Margem média", "Déficit potencial, prometido (% dos dias)", "Déficit potencial, observado (% dos dias)", "Excesso potencial (refeições)", "Déficit potencial (refeições)"]].round(1)

,Decisão,Opção,Margem média,"Déficit potencial, prometido (% dos dias)","Déficit potencial, observado (% dos dias)",Excesso potencial (refeições),Déficit potencial (refeições)
0,preparo,Sem margem,0.0,52.4,58.4,1906.0,1893.0
1,preparo,Margem moderada,17.8,24.2,25.5,3490.0,769.0
2,preparo,Margem alta,35.8,9.7,12.1,5762.0,354.0
3,preparo,Quantitativo da nutricionista,NaN,NaN,39.6,2544.0,994.0
8,compras,Sem margem,0.0,54.6,62.4,1714.0,2113.0
9,compras,Margem moderada,17.3,24.4,32.9,3211.0,968.0
10,compras,Margem alta,37.8,9.9,15.4,5753.0,419.0


## 2. Por que a margem de segurança precisa de contexto

A mesma margem de segurança não protege igual em todo dia. Em dia atípico, o consumo fica muito abaixo do previsto e qualquer margem de segurança só aumenta a diferença. `Tendência da previsão` positiva indica previsão acima do consumo.

In [3]:
t = s.rename(columns={"decisao": "Decisão", "situacao": "Situação do dia", "dias": "Dias", "erro_previsao": "Tendência da previsão (refeições)",
                      "dias_a_menos_sem_folga": "Sem margem", "dias_a_menos_moderada": "Margem moderada", "dias_a_menos_alta": "Margem alta"})
t[["Decisão", "Situação do dia", "Dias", "Tendência da previsão (refeições)", "Sem margem", "Margem moderada", "Margem alta"]].round(1)

,Decisão,Situação do dia,Dias,Tendência da previsão (refeições),Sem margem,Margem moderada,Margem alta
0,preparo,Dia atípico,6,139.2,0.0,0.0,0.0
1,preparo,Dia comum,121,-4.4,62.0,24.0,8.3
2,preparo,Início do período ou volta de pausa,21,21.8,38.1,23.8,19.0
3,preparo,Últimos dias do período,8,24.8,50.0,50.0,50.0
4,compras,Dia atípico,6,135.5,0.0,0.0,0.0
5,compras,Dia comum,121,-7.1,66.1,33.1,12.4
6,compras,Início do período ou volta de pausa,21,19.0,42.9,23.8,19.0
7,compras,Últimos dias do período,8,21.8,50.0,50.0,50.0


## 3. Fichas da próxima emissão

Exemplo do que a nutricionista e a gestora receberiam na sexta-feira mais recente.

In [4]:
arquivo = sorted(SAIDA.glob("fichas_*.md"))[-1]
display(Markdown(arquivo.read_text(encoding="utf-8")))

# Planejamento de refeições: sexta-feira, 30/10/2026

As opções e os dados abaixo apoiam a decisão. Quem define a quantidade é a nutricionista, com a gestora.

## Preparo (semana seguinte)

### Terça-feira, 03/11/2026

**Previsão: 193 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 193 | +0 | 55% |
| Margem moderada | 207 | +14 | 22% |
| Margem alta | 216 | +23 | 10% |

- **Quantitativo da nutricionista:** 210. Combinado com a previsão: 202.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 0 mm no dia, 0 mm de manhã; máxima de 29 °C.
- **Histórico:** Consumo de referência de 194; mesmo dia nas últimas vezes: 192, 185, 199, 201. Na mesma época do ano passado, o consumo ficou em 98% da referência daquele momento.
- **Dias parecidos:** Volta depois de dia sem aula: consumo em torno de 91% da referência (32 dias anteriores).
- **Atenção:** Dias parecidos costumam ficar abaixo da referência: confira antes de usar margem de segurança.

### Quarta-feira, 04/11/2026

**Previsão: 217 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 217 | +0 | 55% |
| Margem moderada | 231 | +14 | 22% |
| Margem alta | 240 | +23 | 10% |

- **Quantitativo da nutricionista:** 230. Combinado com a previsão: 224.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 5 mm no dia, 1 mm de manhã; máxima de 29 °C.
- **Histórico:** Consumo de referência de 200; mesmo dia nas últimas vezes: 232, 202, 193, 215. Na mesma época do ano passado, o consumo ficou em 97% da referência daquele momento.

### Quinta-feira, 05/11/2026

**Previsão: 194 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 194 | +0 | 55% |
| Margem moderada | 208 | +14 | 22% |
| Margem alta | 217 | +23 | 10% |

- **Quantitativo da nutricionista:** 200. Combinado com a previsão: 197.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 7 mm no dia, 2 mm de manhã; máxima de 26 °C.
- **Histórico:** Consumo de referência de 195; mesmo dia nas últimas vezes: 172, 193, 202, 210. Na mesma época do ano passado, o consumo ficou em 97% da referência daquele momento.
- **Dias parecidos:** Quintas-feiras: consumo em torno de 95% da referência (127 dias anteriores).

### Sexta-feira, 06/11/2026

**Previsão: 187 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 187 | +0 | 55% |
| Margem moderada | 201 | +14 | 22% |
| Margem alta | 210 | +23 | 10% |

- **Quantitativo da nutricionista:** 190. Combinado com a previsão: 188.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 6 mm no dia, 0 mm de manhã; máxima de 26 °C.
- **Histórico:** Consumo de referência de 190; mesmo dia nas últimas vezes: 149, 181, 194, 201. Na mesma época do ano passado, o consumo ficou em 97% da referência daquele momento.

## Compras (daqui a duas semanas)

### Segunda-feira, 09/11/2026

**Previsão: 214 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 214 | +0 | 55% |
| Margem moderada | 234 | +20 | 25% |
| Margem alta | 246 | +32 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 2 mm no dia, 1 mm de manhã; máxima de 22 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 202; mesmo dia nas últimas vezes: 199, 224, 201, 237. Na mesma época do ano passado, o consumo ficou em 95% da referência daquele momento.

### Terça-feira, 10/11/2026

**Previsão: 197 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 197 | +0 | 55% |
| Margem moderada | 217 | +20 | 25% |
| Margem alta | 229 | +32 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 1 mm no dia, 0 mm de manhã; máxima de 22 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 194; mesmo dia nas últimas vezes: 192, 185, 199, 201. Na mesma época do ano passado, o consumo ficou em 93% da referência daquele momento.

### Quarta-feira, 11/11/2026

**Previsão: 215 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 215 | +0 | 55% |
| Margem moderada | 235 | +20 | 25% |
| Margem alta | 247 | +32 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 21 mm no dia, 1 mm de manhã; máxima de 23 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 200; mesmo dia nas últimas vezes: 232, 202, 193, 215. Na mesma época do ano passado, o consumo ficou em 94% da referência daquele momento.

### Quinta-feira, 12/11/2026

**Previsão: 197 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 197 | +0 | 55% |
| Margem moderada | 217 | +20 | 25% |
| Margem alta | 229 | +32 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 9 mm no dia, 1 mm de manhã; máxima de 24 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 195; mesmo dia nas últimas vezes: 172, 193, 202, 210. Na mesma época do ano passado, o consumo ficou em 93% da referência daquele momento.
- **Dias parecidos:** Quintas-feiras: consumo em torno de 95% da referência (127 dias anteriores).

### Sexta-feira, 13/11/2026

**Previsão: 194 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 194 | +0 | 55% |
| Margem moderada | 214 | +20 | 25% |
| Margem alta | 226 | +32 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 6 mm no dia, 1 mm de manhã; máxima de 26 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 190; mesmo dia nas últimas vezes: 149, 181, 194, 201. Na mesma época do ano passado, o consumo ficou em 93% da referência daquele momento.
